# HEART: Hierarchical Embedding for Affective and Riemannian Trajectories

In [29]:
import os
import pickle
import numpy as np
from collections import defaultdict
from sklearn.covariance import OAS

fs = 250
window_sec = 1
window_samples = fs * window_sec
n_channels = 32
data_dir = "Processed_data"

emotion_labels = [
    "Anger", "Anger", "Anger",
    "Disgust", "Disgust", "Disgust",
    "Fear", "Fear", "Fear",
    "Sadness", "Sadness", "Sadness",
    "Neutral", "Neutral", "Neutral", "Neutral",
    "Amusement", "Amusement", "Amusement",
    "Inspiration", "Inspiration", "Inspiration",
    "Joy", "Joy", "Joy",
    "Tenderness", "Tenderness", "Tenderness"
]

def compute_covariance_oas(X):
    oas = OAS()
    oas.fit(X.T)
    cov = oas.covariance_
    cov += 1e-6 * np.eye(cov.shape[0])
    return cov

def extract_cov_features(eeg_trial, window_samples):
    n_channels, n_samples = eeg_trial.shape
    n_windows = n_samples // window_samples
    features = []
    for w in range(n_windows):
        start = w * window_samples
        end = start + window_samples
        segment = eeg_trial[:, start:end]
        cov = compute_covariance_oas(segment)
        iu = np.triu_indices(n_channels)
        features.append(cov[iu])
    return np.array(features)

def find_change_point_2segments(features):
    # Compute distances between consecutive windows
    distances = np.linalg.norm(np.diff(features, axis=0), axis=1)
    # Find index of max jump (change point)
    change_idx = np.argmax(distances) + 1  # +1 to mark start of second segment
    return change_idx

# Store boundaries for each emotion and trial
emotion_boundaries = defaultdict(list)

for filename in sorted(os.listdir(data_dir)):
    if not filename.endswith(".pkl"):
        continue
    with open(os.path.join(data_dir, filename), "rb") as f:
        eeg_data = pickle.load(f)

    for trial_idx, trial in enumerate(eeg_data[:28]):
        emotion = emotion_labels[trial_idx]
        features = extract_cov_features(trial, window_samples)
        if len(features) < 2:
            continue  # Not enough windows to split

        boundary = find_change_point_2segments(features)
        # Convert boundary to time in seconds
        boundary_time_sec = boundary * window_sec
        emotion_boundaries[emotion].append(boundary_time_sec)

# Summary: average and std boundary times per emotion
print("Segment boundaries (seconds) for 2-segment split per emotion:")
for emotion, boundaries in emotion_boundaries.items():
    avg_b = np.mean(boundaries)
    std_b = np.std(boundaries)
    print(f"{emotion:12s}: Mean boundary = {avg_b:.2f}s ± {std_b:.2f}s (from {len(boundaries)} trials)")

Segment boundaries (seconds) for 2-segment split per emotion:
Anger       : Mean boundary = 15.32s ± 8.87s (from 369 trials)
Disgust     : Mean boundary = 16.25s ± 8.50s (from 369 trials)
Fear        : Mean boundary = 14.47s ± 8.15s (from 369 trials)
Sadness     : Mean boundary = 16.83s ± 8.81s (from 369 trials)
Neutral     : Mean boundary = 13.15s ± 8.23s (from 492 trials)
Amusement   : Mean boundary = 19.45s ± 7.86s (from 369 trials)
Inspiration : Mean boundary = 17.02s ± 8.38s (from 369 trials)
Joy         : Mean boundary = 12.35s ± 8.57s (from 369 trials)
Tenderness  : Mean boundary = 15.67s ± 8.96s (from 369 trials)


In [3]:
import os
import pickle
import numpy as np
from collections import defaultdict

fs = 250
n_channels = 32
data_dir = "Processed_data"
output_path = "heart_features_with_subjects_2.pkl"

# Emotion labels (trial order known)
emotion_labels = [
    "Anger", "Anger", "Anger",
    "Disgust", "Disgust", "Disgust",
    "Fear", "Fear", "Fear",
    "Sadness", "Sadness", "Sadness",
    "Neutral", "Neutral", "Neutral", "Neutral",
    "Amusement", "Amusement", "Amusement",
    "Inspiration", "Inspiration", "Inspiration",
    "Joy", "Joy", "Joy",
    "Tenderness", "Tenderness", "Tenderness"
]

# Precomputed mean segment boundary per emotion (in seconds)
emotion_mean_boundaries = {
    "Anger": 15.32,
    "Disgust": 16.25,
    "Fear": 14.47,
    "Sadness": 16.83,
    "Neutral": 13.15,
    "Amusement": 19.45,
    "Inspiration": 17.02,
    "Joy": 12.35,
    "Tenderness": 15.67
}

# --- Placeholder functions (replace with your implementations) ---
def compute_covariance(X):
    from sklearn.covariance import OAS
    oas = OAS()
    oas.fit(X.T)
    cov = oas.covariance_
    cov += 1e-6 * np.eye(cov.shape[0])
    return cov

def geodesic_distance(C1, C2):
    from scipy.linalg import logm
    sqrt_C1 = np.linalg.cholesky(C1)
    inv_sqrt_C1 = np.linalg.inv(sqrt_C1)
    C2_norm = inv_sqrt_C1 @ C2 @ inv_sqrt_C1.T
    log_C2_norm = logm(C2_norm)
    return np.linalg.norm(log_C2_norm, 'fro')

def tangent_space_features(C, ref=None):
    from scipy.linalg import logm
    if ref is None:
        ref = np.eye(n_channels)
    logm_diff = logm(np.linalg.inv(ref) @ C)
    iu = np.triu_indices_from(logm_diff)
    return logm_diff[iu]

# --- HEART feature extraction ---
def extract_heart_features_by_boundary(eeg_trial, boundary_sec):
    boundary_sample = int(boundary_sec * fs)
    C1 = compute_covariance(eeg_trial[:, :boundary_sample])
    C2 = compute_covariance(eeg_trial[:, boundary_sample:])
    
    d12 = geodesic_distance(C1, C2)
    speed = d12 / (30 - boundary_sec)
    curvature = 0  # Not defined with only 2 matrices

    t1 = tangent_space_features(C1)
    t2 = tangent_space_features(C2)

    return np.concatenate([t1, t2, [d12, speed, curvature]])

# --- Feature extraction loop ---
X = []
y = []
subjects = []

for filename in sorted(os.listdir(data_dir)):
    if not filename.endswith(".pkl"):
        continue

    subject_id = filename.split(".")[0]
    with open(os.path.join(data_dir, filename), "rb") as f:
        eeg_data = pickle.load(f)

    for trial_idx, trial in enumerate(eeg_data[:28]):
        if trial.shape[1] != 7500:
            continue

        emotion = emotion_labels[trial_idx]
        boundary = emotion_mean_boundaries.get(emotion, 15.0)

        features = extract_heart_features_by_boundary(trial, boundary)
        X.append(features)
        y.append(emotion)
        subjects.append(subject_id)

# --- Save to pickle ---
with open(output_path, "wb") as f:
    pickle.dump({"features": X, "labels": y, "subjects": subjects}, f)

print(f"Saved features to {output_path} | Total samples: {len(X)}")

Saved features to heart_features_with_subjects_2.pkl | Total samples: 3444


In [4]:
import pickle

# Load previously saved data
with open("heart_features_with_subjects_2.pkl", "rb") as f:
    data = pickle.load(f)

X = data["features"]
y = data["labels"]
subjects = data["subjects"]

# Russell’s Circumplex Model: Valence (V), Arousal (A), Dominance (D)
emotion_to_vad = {
    "Anger":       [0.2, 0.9, 0.8],
    "Disgust":     [0.2, 0.6, 0.5],
    "Fear":        [0.1, 0.9, 0.3],
    "Sadness":     [0.1, 0.2, 0.3],
    "Neutral":     [0.5, 0.5, 0.5],
    "Amusement":   [0.8, 0.8, 0.6],
    "Inspiration": [0.9, 0.6, 0.7],
    "Joy":         [0.9, 0.9, 0.8],
    "Tenderness":  [0.8, 0.4, 0.6]
}

# Create VAD arrays
V = []
A = []
D = []

for emotion in y:
    valence, arousal, dominance = emotion_to_vad.get(emotion, [0.5, 0.5, 0.5])  # default neutral
    V.append(valence)
    A.append(arousal)
    D.append(dominance)

# Save augmented data
augmented_data = {
    "features": X,
    "labels": y,
    "subjects": subjects,
    "valence": V,
    "arousal": A,
    "dominance": D
}

with open("heart_features_vad_2.pkl", "wb") as f:
    pickle.dump(augmented_data, f)

print("Saved augmented data with VAD to heart_features_with_vad.pkl")

Saved augmented data with VAD to heart_features_with_vad.pkl


In [31]:
import pickle
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# Load data
with open("heart_features_vad.pkl", "rb") as f:
    data = pickle.load(f)

X = np.array(data["features"])
V = np.array(data["valence"])
A = np.array(data["arousal"])
D = np.array(data["dominance"])
emotions = np.array(data["labels"])
subjects = np.array(data["subjects"])

# Encode emotions
le_emotion = LabelEncoder()
y_emotion = le_emotion.fit_transform(emotions)

# Split by subject
unique_subjects = np.unique(subjects)
train_subjects, test_subjects = train_test_split(unique_subjects, test_size=0.2, random_state=42)
train_mask = np.isin(subjects, train_subjects)
test_mask = np.isin(subjects, test_subjects)

X_train, X_test = X[train_mask], X[test_mask]
V_train, V_test = V[train_mask], V[test_mask]
A_train, A_test = A[train_mask], A[test_mask]
D_train, D_test = D[train_mask], D[test_mask]
y_train, y_test = y_emotion[train_mask], y_emotion[test_mask]

# Train regressors for V, A, D on train set
reg_V = RandomForestRegressor(n_estimators=100, random_state=42)
reg_A = RandomForestRegressor(n_estimators=100, random_state=42)
reg_D = RandomForestRegressor(n_estimators=100, random_state=42)

reg_V.fit(X_train, V_train)
reg_A.fit(X_train, A_train)
reg_D.fit(X_train, D_train)

# Train base emotion classifier directly on EEG features
clf_emotion_base = RandomForestClassifier(n_estimators=100, random_state=42)
clf_emotion_base.fit(X_train, y_train)

# Predict on test set with base models
V_pred = reg_V.predict(X_test).reshape(-1, 1)
A_pred = reg_A.predict(X_test).reshape(-1, 1)
D_pred = reg_D.predict(X_test).reshape(-1, 1)
emotion_pred_proba = clf_emotion_base.predict_proba(X_test)  # use probabilities for richer info

# Prepare meta-features for meta-classifier
meta_features_test = np.hstack([V_pred, A_pred, D_pred, emotion_pred_proba])

# Similarly, get meta-features for training (using cross-validation or train predictions)
# For simplicity, here we use predictions on training set itself
V_train_pred = reg_V.predict(X_train).reshape(-1, 1)
A_train_pred = reg_A.predict(X_train).reshape(-1, 1)
D_train_pred = reg_D.predict(X_train).reshape(-1, 1)
emotion_train_pred_proba = clf_emotion_base.predict_proba(X_train)

meta_features_train = np.hstack([V_train_pred, A_train_pred, D_train_pred, emotion_train_pred_proba])

# Train meta-classifier on meta-features
meta_clf = RandomForestClassifier(n_estimators=100, random_state=42)
meta_clf.fit(meta_features_train, y_train)

# Meta-classifier predicts final emotion labels
y_pred_meta = meta_clf.predict(meta_features_test)

# Evaluate meta-classifier
print("Meta-classifier Emotion classification report:")
print(classification_report(y_test, y_pred_meta, target_names=le_emotion.classes_))

Meta-classifier Emotion classification report:
              precision    recall  f1-score   support

   Amusement       1.00      0.99      0.99        75
       Anger       0.81      0.87      0.84        75
     Disgust       0.91      0.92      0.91        75
        Fear       0.94      0.83      0.88        75
 Inspiration       0.77      0.77      0.77        75
         Joy       0.94      0.87      0.90        75
     Neutral       0.92      0.94      0.93       100
     Sadness       0.74      0.77      0.76        75
  Tenderness       0.81      0.87      0.84        75

    accuracy                           0.87       700
   macro avg       0.87      0.87      0.87       700
weighted avg       0.87      0.87      0.87       700



In [25]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# --- Results ---
print("Meta-Classifier Performance:\n")
print(classification_report(y_test, y_pred_meta, target_names=le.classes_))

# --- Additional Metrics ---
# Average accuracy
accuracy = accuracy_score(y_test, y_pred_meta) * 100
print(f"Average Accuracy: {accuracy:.2f}%")

# Standard deviation of accuracy (over folds)
# Note: Since we used a hold-out split for test, not CV, std isn't directly meaningful.
# Instead, we can compute accuracy per class and take std.
class_accuracies = []
for i in range(len(le.classes_)):
    class_mask = y_test == i
    class_acc = accuracy_score(y_test[class_mask], y_pred_meta[class_mask])
    class_accuracies.append(class_acc)

std_accuracy = np.std(class_accuracies) * 100
print(f"Standard Deviation of Per-Class Accuracy: ±{std_accuracy:.2f}%")

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred_meta)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(xticks_rotation=45, cmap="Blues")

# Save as PDF
plt.tight_layout()
plt.savefig("confusion_matrix_meta.pdf", format="pdf")
plt.close()

Meta-Classifier Performance:

              precision    recall  f1-score   support

   Amusement       1.00      0.99      0.99        75
       Anger       0.84      0.92      0.88        75
     Disgust       0.91      0.93      0.92        75
        Fear       0.97      0.85      0.91        75
 Inspiration       0.86      0.80      0.83        75
         Joy       1.00      0.96      0.98        75
     Neutral       0.98      0.99      0.99       100
     Sadness       0.80      0.85      0.83        75
  Tenderness       0.86      0.89      0.88        75

    accuracy                           0.91       700
   macro avg       0.91      0.91      0.91       700
weighted avg       0.92      0.91      0.91       700

Average Accuracy: 91.29%
Standard Deviation of Per-Class Accuracy: ±6.17%


In [26]:
import pickle
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import classification_report

# --- Load preprocessed data ---
with open("heart_features_vad.pkl", "rb") as f:
    data = pickle.load(f)

X = np.array(data["features"])
V = np.array(data["valence"])
A = np.array(data["arousal"])
D = np.array(data["dominance"])
labels = np.array(data["labels"])
subjects = np.array(data["subjects"])

# Define positive, negative, and neutral emotion lists
positive_emotions = ["Joy", "Amusement", "Inspiration", "Tenderness"]
negative_emotions = ["Anger", "Disgust", "Fear", "Sadness"]
neutral_emotion = "Neutral"

# Filter out neutral samples and create binary labels:
mask = (labels != neutral_emotion)
X = X[mask]
V = V[mask]
A = A[mask]
D = D[mask]
labels = labels[mask]
subjects = subjects[mask]

# Create binary labels: 1 for positive, 0 for negative
binary_labels = []
for label in labels:
    if label in positive_emotions:
        binary_labels.append(1)
    elif label in negative_emotions:
        binary_labels.append(0)
    else:
        # This should not happen due to mask, but just in case
        binary_labels.append(-1)

y = np.array(binary_labels)

# Encode labels (optional here, but just keep consistent)
le = LabelEncoder()
y = le.fit_transform(y)

# Split by subject (80/20 holdout for testing)
unique_subjects = np.unique(subjects)
train_subjects, test_subjects = train_test_split(unique_subjects, test_size=0.2, random_state=42)
train_mask = np.isin(subjects, train_subjects)
test_mask = np.isin(subjects, test_subjects)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
V_train, V_test = V[train_mask], V[test_mask]
A_train, A_test = A[train_mask], A[test_mask]
D_train, D_test = D[train_mask], D[test_mask]
subjects_train = subjects[train_mask]

# --- Prepare meta-training data via GroupKFold ---
# Now number of classes = 2
n_classes = 2
meta_train = np.zeros((X_train.shape[0], 3 + n_classes))  # V, A, D + class probs

gkf = GroupKFold(n_splits=5)
for train_idx, val_idx in gkf.split(X_train, y_train, groups=subjects_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    V_fold_train, V_fold_val = V_train[train_idx], V_train[val_idx]
    A_fold_train, A_fold_val = A_train[train_idx], A_train[val_idx]
    D_fold_train, D_fold_val = D_train[train_idx], D_train[val_idx]

    # Base models
    reg_V = RandomForestRegressor(n_estimators=100, random_state=42)
    reg_A = RandomForestRegressor(n_estimators=100, random_state=42)
    reg_D = RandomForestRegressor(n_estimators=100, random_state=42)
    clf_emotion = RandomForestClassifier(n_estimators=100, random_state=42)

    reg_V.fit(X_fold_train, V_fold_train)
    reg_A.fit(X_fold_train, A_fold_train)
    reg_D.fit(X_fold_train, D_fold_train)
    clf_emotion.fit(X_fold_train, y_fold_train)

    # Predictions for validation fold
    V_pred = reg_V.predict(X_fold_val).reshape(-1, 1)
    A_pred = reg_A.predict(X_fold_val).reshape(-1, 1)
    D_pred = reg_D.predict(X_fold_val).reshape(-1, 1)
    probs = clf_emotion.predict_proba(X_fold_val)

    meta_train[val_idx] = np.hstack([V_pred, A_pred, D_pred, probs])

# --- Train final base models for test prediction ---
reg_V_final = RandomForestRegressor(n_estimators=100, random_state=42)
reg_A_final = RandomForestRegressor(n_estimators=100, random_state=42)
reg_D_final = RandomForestRegressor(n_estimators=100, random_state=42)
clf_emotion_final = RandomForestClassifier(n_estimators=100, random_state=42)

reg_V_final.fit(X_train, V_train)
reg_A_final.fit(X_train, A_train)
reg_D_final.fit(X_train, D_train)
clf_emotion_final.fit(X_train, y_train)

# Predict meta-features for test set
V_pred_test = reg_V_final.predict(X_test).reshape(-1, 1)
A_pred_test = reg_A_final.predict(X_test).reshape(-1, 1)
D_pred_test = reg_D_final.predict(X_test).reshape(-1, 1)
probs_test = clf_emotion_final.predict_proba(X_test)

meta_test = np.hstack([V_pred_test, A_pred_test, D_pred_test, probs_test])

# --- Train and evaluate meta-classifier ---
meta_clf = RandomForestClassifier(n_estimators=100, random_state=42)
meta_clf.fit(meta_train, y_train)
y_pred_meta = meta_clf.predict(meta_test)

# --- Results ---
print("Meta-Classifier Binary Performance:\n")
print(classification_report(y_test, y_pred_meta, target_names=["Negative", "Positive"]))

Meta-Classifier Binary Performance:

              precision    recall  f1-score   support

    Negative       0.91      0.89      0.90       300
    Positive       0.89      0.91      0.90       300

    accuracy                           0.90       600
   macro avg       0.90      0.90      0.90       600
weighted avg       0.90      0.90      0.90       600



### Channel reduction

In [6]:
import os
import pickle
import numpy as np
from collections import defaultdict

fs = 250
data_dir = "Processed_data"
output_path = "heart_features_with_subjects_20.pkl"

# Emotion labels (trial order known)
emotion_labels = [
    "Anger", "Anger", "Anger",
    "Disgust", "Disgust", "Disgust",
    "Fear", "Fear", "Fear",
    "Sadness", "Sadness", "Sadness",
    "Neutral", "Neutral", "Neutral", "Neutral",
    "Amusement", "Amusement", "Amusement",
    "Inspiration", "Inspiration", "Inspiration",
    "Joy", "Joy", "Joy",
    "Tenderness", "Tenderness", "Tenderness"
]

# Precomputed mean segment boundary per emotion (in seconds)
emotion_mean_boundaries = {
    "Anger": 15.32,
    "Disgust": 16.25,
    "Fear": 14.47,
    "Sadness": 16.83,
    "Neutral": 13.15,
    "Amusement": 19.45,
    "Inspiration": 17.02,
    "Joy": 12.35,
    "Tenderness": 15.67
}

# --- Channel setup ---
all_channels = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FC1", "FC2", "FC5",
    "FC6", "Cz", "C3", "C4", "T7", "T8", "CP1", "CP2", "CP5", "CP6",
    "Pz", "P3", "P4", "P7", "P8", "PO3", "PO4", "Oz", "O1", "O2",
    "HEOL", "HEOR"
]

selected_channels = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8",
    "FC1", "FC2", "FC5", "FC6", "Cz", "C3", "C4",
    "T7", "T8", "P7", "P8", "O1", "O2"
]

channel_indices_20 = [all_channels.index(ch) for ch in selected_channels]

# --- Feature extraction functions ---
def compute_covariance(X):
    from sklearn.covariance import OAS
    oas = OAS()
    oas.fit(X.T)
    cov = oas.covariance_
    cov += 1e-6 * np.eye(cov.shape[0])
    return cov

def geodesic_distance(C1, C2):
    from scipy.linalg import logm
    sqrt_C1 = np.linalg.cholesky(C1)
    inv_sqrt_C1 = np.linalg.inv(sqrt_C1)
    C2_norm = inv_sqrt_C1 @ C2 @ inv_sqrt_C1.T
    log_C2_norm = logm(C2_norm)
    return np.linalg.norm(log_C2_norm, 'fro')

def tangent_space_features(C, ref=None):
    from scipy.linalg import logm
    if ref is None:
        ref = C
    logm_diff = logm(np.linalg.inv(ref) @ C)
    iu = np.triu_indices_from(logm_diff)
    return logm_diff[iu]

def extract_heart_features_by_boundary(eeg_trial, boundary_sec):
    boundary_sample = int(boundary_sec * fs)
    C1 = compute_covariance(eeg_trial[:, :boundary_sample])
    C2 = compute_covariance(eeg_trial[:, boundary_sample:])

    d12 = geodesic_distance(C1, C2)
    speed = d12 / (30 - boundary_sec)
    curvature = 0  # Not defined with only 2 matrices

    t1 = tangent_space_features(C1)
    t2 = tangent_space_features(C2)

    return np.concatenate([t1, t2, [d12, speed, curvature]])

# --- Feature extraction loop ---
X = []
y = []
subjects = []

for filename in sorted(os.listdir(data_dir)):
    if not filename.endswith(".pkl"):
        continue

    subject_id = filename.split(".")[0]
    with open(os.path.join(data_dir, filename), "rb") as f:
        eeg_data = pickle.load(f)

    for trial_idx, trial in enumerate(eeg_data[:28]):
        if trial.shape[1] != 7500:
            continue

        emotion = emotion_labels[trial_idx]
        boundary = emotion_mean_boundaries.get(emotion, 15.0)

        # Select 20-channel configuration
        trial_20 = trial[channel_indices_20, :]

        features = extract_heart_features_by_boundary(trial_20, boundary)
        X.append(features)
        y.append(emotion)
        subjects.append(subject_id)

# --- Save features to pickle ---
with open(output_path, "wb") as f:
    pickle.dump({"features": X, "labels": y, "subjects": subjects}, f)

print(f"Saved features to {output_path} | Total samples: {len(X)}")

Saved features to heart_features_with_subjects_20.pkl | Total samples: 3444


In [7]:
import pickle

# Load previously saved data
with open("heart_features_with_subjects_20.pkl", "rb") as f:
    data = pickle.load(f)

X = data["features"]
y = data["labels"]
subjects = data["subjects"]

# Russell’s Circumplex Model: Valence (V), Arousal (A), Dominance (D)
emotion_to_vad = {
    "Anger":       [0.2, 0.9, 0.8],
    "Disgust":     [0.2, 0.6, 0.5],
    "Fear":        [0.1, 0.9, 0.3],
    "Sadness":     [0.1, 0.2, 0.3],
    "Neutral":     [0.5, 0.5, 0.5],
    "Amusement":   [0.8, 0.8, 0.6],
    "Inspiration": [0.9, 0.6, 0.7],
    "Joy":         [0.9, 0.9, 0.8],
    "Tenderness":  [0.8, 0.4, 0.6]
}

# Create VAD arrays
V = []
A = []
D = []

for emotion in y:
    valence, arousal, dominance = emotion_to_vad.get(emotion, [0.5, 0.5, 0.5])  # default neutral
    V.append(valence)
    A.append(arousal)
    D.append(dominance)

# Save augmented data
augmented_data = {
    "features": X,
    "labels": y,
    "subjects": subjects,
    "valence": V,
    "arousal": A,
    "dominance": D
}

with open("heart_features_vad_20.pkl", "wb") as f:
    pickle.dump(augmented_data, f)

print("Saved augmented data with VAD to heart_features_vad_20.pkl")

Saved augmented data with VAD to heart_features_with_vad_20.pkl


In [11]:
import pickle
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import classification_report

# --- Load preprocessed data ---
with open("heart_features_vad_20.pkl", "rb") as f:
    data = pickle.load(f)

X = np.array(data["features"])
V = np.array(data["valence"])
A = np.array(data["arousal"])
D = np.array(data["dominance"])
labels = np.array(data["labels"])
subjects = np.array(data["subjects"])

# Encode emotion labels
le = LabelEncoder()
y = le.fit_transform(labels)

# Split by subject (80/20 holdout for testing)
unique_subjects = np.unique(subjects)
train_subjects, test_subjects = train_test_split(unique_subjects, test_size=0.2, random_state=42)
train_mask = np.isin(subjects, train_subjects)
test_mask = np.isin(subjects, test_subjects)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
V_train, V_test = V[train_mask], V[test_mask]
A_train, A_test = A[train_mask], A[test_mask]
D_train, D_test = D[train_mask], D[test_mask]
subjects_train = subjects[train_mask]

# --- Prepare meta-training data via GroupKFold ---
n_classes = len(le.classes_)
meta_train = np.zeros((X_train.shape[0], 3 + n_classes))  # V, A, D + class probs

gkf = GroupKFold(n_splits=5)
for train_idx, val_idx in gkf.split(X_train, y_train, groups=subjects_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    V_fold_train, V_fold_val = V_train[train_idx], V_train[val_idx]
    A_fold_train, A_fold_val = A_train[train_idx], A_train[val_idx]
    D_fold_train, D_fold_val = D_train[train_idx], D_train[val_idx]

    # Base models
    reg_V = RandomForestRegressor(n_estimators=100, random_state=42)
    reg_A = RandomForestRegressor(n_estimators=100, random_state=42)
    reg_D = RandomForestRegressor(n_estimators=100, random_state=42)
    clf_emotion = RandomForestClassifier(n_estimators=100, random_state=42)

    reg_V.fit(X_fold_train, V_fold_train)
    reg_A.fit(X_fold_train, A_fold_train)
    reg_D.fit(X_fold_train, D_fold_train)
    clf_emotion.fit(X_fold_train, y_fold_train)

    # Predictions for validation fold
    V_pred = reg_V.predict(X_fold_val).reshape(-1, 1)
    A_pred = reg_A.predict(X_fold_val).reshape(-1, 1)
    D_pred = reg_D.predict(X_fold_val).reshape(-1, 1)
    probs = clf_emotion.predict_proba(X_fold_val)

    meta_train[val_idx] = np.hstack([V_pred, A_pred, D_pred, probs])

# --- Train final base models for test prediction ---
reg_V_final = RandomForestRegressor(n_estimators=100, random_state=42)
reg_A_final = RandomForestRegressor(n_estimators=100, random_state=42)
reg_D_final = RandomForestRegressor(n_estimators=100, random_state=42)
clf_emotion_final = RandomForestClassifier(n_estimators=100, random_state=42)

reg_V_final.fit(X_train, V_train)
reg_A_final.fit(X_train, A_train)
reg_D_final.fit(X_train, D_train)
clf_emotion_final.fit(X_train, y_train)

# Predict meta-features for test set
V_pred_test = reg_V_final.predict(X_test).reshape(-1, 1)
A_pred_test = reg_A_final.predict(X_test).reshape(-1, 1)
D_pred_test = reg_D_final.predict(X_test).reshape(-1, 1)
probs_test = clf_emotion_final.predict_proba(X_test)

meta_test = np.hstack([V_pred_test, A_pred_test, D_pred_test, probs_test])

# --- Train and evaluate meta-classifier ---
meta_clf = RandomForestClassifier(n_estimators=100, random_state=42)
meta_clf.fit(meta_train, y_train)
y_pred_meta = meta_clf.predict(meta_test)

# --- Results ---
print("Meta-Classifier Performance:\n")
print(classification_report(y_test, y_pred_meta, target_names=le.classes_))

Meta-Classifier Performance:

              precision    recall  f1-score   support

   Amusement       1.00      0.97      0.99        75
       Anger       0.89      0.88      0.89        75
     Disgust       0.95      0.96      0.95        75
        Fear       0.95      0.96      0.95        75
 Inspiration       0.90      0.80      0.85        75
         Joy       0.99      0.92      0.95        75
     Neutral       0.96      0.99      0.98       100
     Sadness       0.82      0.92      0.87        75
  Tenderness       0.88      0.91      0.89        75

    accuracy                           0.93       700
   macro avg       0.93      0.92      0.92       700
weighted avg       0.93      0.93      0.93       700



In [9]:
import os
import pickle
import numpy as np
from collections import defaultdict

fs = 250
data_dir = "Processed_data"
output_path = "heart_features_with_subjects_8.pkl"

# Emotion labels (trial order known)
emotion_labels = [
    "Anger", "Anger", "Anger",
    "Disgust", "Disgust", "Disgust",
    "Fear", "Fear", "Fear",
    "Sadness", "Sadness", "Sadness",
    "Neutral", "Neutral", "Neutral", "Neutral",
    "Amusement", "Amusement", "Amusement",
    "Inspiration", "Inspiration", "Inspiration",
    "Joy", "Joy", "Joy",
    "Tenderness", "Tenderness", "Tenderness"
]

# Mean segment boundaries
emotion_mean_boundaries = {
    "Anger": 15.32,
    "Disgust": 16.25,
    "Fear": 14.47,
    "Sadness": 16.83,
    "Neutral": 13.15,
    "Amusement": 19.45,
    "Inspiration": 17.02,
    "Joy": 12.35,
    "Tenderness": 15.67
}

# --- Channel setup ---
all_channels = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FC1", "FC2", "FC5",
    "FC6", "Cz", "C3", "C4", "T7", "T8", "CP1", "CP2", "CP5", "CP6",
    "Pz", "P3", "P4", "P7", "P8", "PO3", "PO4", "Oz", "O1", "O2",
    "HEOL", "HEOR"
]

# 8-channel configuration
selected_channels = ["Fp1", "Fp2", "Fz", "Cz", "P3", "P4", "O1", "O2"]
channel_indices_8 = [all_channels.index(ch) for ch in selected_channels]

# --- Feature extraction functions ---
def compute_covariance(X):
    from sklearn.covariance import OAS
    oas = OAS()
    oas.fit(X.T)
    cov = oas.covariance_
    cov += 1e-6 * np.eye(cov.shape[0])
    return cov

def geodesic_distance(C1, C2):
    from scipy.linalg import logm
    sqrt_C1 = np.linalg.cholesky(C1)
    inv_sqrt_C1 = np.linalg.inv(sqrt_C1)
    C2_norm = inv_sqrt_C1 @ C2 @ inv_sqrt_C1.T
    log_C2_norm = logm(C2_norm)
    return np.linalg.norm(log_C2_norm, 'fro')

def tangent_space_features(C, ref=None):
    from scipy.linalg import logm
    if ref is None:
        ref = C
    logm_diff = logm(np.linalg.inv(ref) @ C)
    iu = np.triu_indices_from(logm_diff)
    return logm_diff[iu]

def extract_heart_features_by_boundary(eeg_trial, boundary_sec):
    boundary_sample = int(boundary_sec * fs)
    C1 = compute_covariance(eeg_trial[:, :boundary_sample])
    C2 = compute_covariance(eeg_trial[:, boundary_sample:])

    d12 = geodesic_distance(C1, C2)
    speed = d12 / (30 - boundary_sec)
    curvature = 0  # Not defined with only 2 matrices

    t1 = tangent_space_features(C1)
    t2 = tangent_space_features(C2)

    return np.concatenate([t1, t2, [d12, speed, curvature]])

# --- Feature extraction loop ---
X = []
y = []
subjects = []

for filename in sorted(os.listdir(data_dir)):
    if not filename.endswith(".pkl"):
        continue

    subject_id = filename.split(".")[0]
    with open(os.path.join(data_dir, filename), "rb") as f:
        eeg_data = pickle.load(f)

    for trial_idx, trial in enumerate(eeg_data[:28]):
        if trial.shape[1] != 7500:
            continue

        emotion = emotion_labels[trial_idx]
        boundary = emotion_mean_boundaries.get(emotion, 15.0)

        # Select only 8 channels
        trial_8 = trial[channel_indices_8, :]

        features = extract_heart_features_by_boundary(trial_8, boundary)
        X.append(features)
        y.append(emotion)
        subjects.append(subject_id)

# --- Save to pickle ---
with open(output_path, "wb") as f:
    pickle.dump({"features": X, "labels": y, "subjects": subjects}, f)

print(f"Saved 8-channel features to {output_path} | Total samples: {len(X)}")

Saved 8-channel features to heart_features_with_subjects_8.pkl | Total samples: 3444


In [10]:
import pickle

# Load previously saved data
with open("heart_features_with_subjects_8.pkl", "rb") as f:
    data = pickle.load(f)

X = data["features"]
y = data["labels"]
subjects = data["subjects"]

# Russell’s Circumplex Model: Valence (V), Arousal (A), Dominance (D)
emotion_to_vad = {
    "Anger":       [0.2, 0.9, 0.8],
    "Disgust":     [0.2, 0.6, 0.5],
    "Fear":        [0.1, 0.9, 0.3],
    "Sadness":     [0.1, 0.2, 0.3],
    "Neutral":     [0.5, 0.5, 0.5],
    "Amusement":   [0.8, 0.8, 0.6],
    "Inspiration": [0.9, 0.6, 0.7],
    "Joy":         [0.9, 0.9, 0.8],
    "Tenderness":  [0.8, 0.4, 0.6]
}

# Create VAD arrays
V = []
A = []
D = []

for emotion in y:
    valence, arousal, dominance = emotion_to_vad.get(emotion, [0.5, 0.5, 0.5])  # default neutral
    V.append(valence)
    A.append(arousal)
    D.append(dominance)

# Save augmented data
augmented_data = {
    "features": X,
    "labels": y,
    "subjects": subjects,
    "valence": V,
    "arousal": A,
    "dominance": D
}

with open("heart_features_vad_8.pkl", "wb") as f:
    pickle.dump(augmented_data, f)

print("Saved augmented data with VAD to heart_features_vad_8.pkl")

Saved augmented data with VAD to heart_features_vad_8.pkl


In [12]:
import pickle
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import classification_report

# --- Load preprocessed data ---
with open("heart_features_vad_8.pkl", "rb") as f:
    data = pickle.load(f)

X = np.array(data["features"])
V = np.array(data["valence"])
A = np.array(data["arousal"])
D = np.array(data["dominance"])
labels = np.array(data["labels"])
subjects = np.array(data["subjects"])

# Encode emotion labels
le = LabelEncoder()
y = le.fit_transform(labels)

# Split by subject (80/20 holdout for testing)
unique_subjects = np.unique(subjects)
train_subjects, test_subjects = train_test_split(unique_subjects, test_size=0.2, random_state=42)
train_mask = np.isin(subjects, train_subjects)
test_mask = np.isin(subjects, test_subjects)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
V_train, V_test = V[train_mask], V[test_mask]
A_train, A_test = A[train_mask], A[test_mask]
D_train, D_test = D[train_mask], D[test_mask]
subjects_train = subjects[train_mask]

# --- Prepare meta-training data via GroupKFold ---
n_classes = len(le.classes_)
meta_train = np.zeros((X_train.shape[0], 3 + n_classes))  # V, A, D + class probs

gkf = GroupKFold(n_splits=5)
for train_idx, val_idx in gkf.split(X_train, y_train, groups=subjects_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    V_fold_train, V_fold_val = V_train[train_idx], V_train[val_idx]
    A_fold_train, A_fold_val = A_train[train_idx], A_train[val_idx]
    D_fold_train, D_fold_val = D_train[train_idx], D_train[val_idx]

    # Base models
    reg_V = RandomForestRegressor(n_estimators=100, random_state=42)
    reg_A = RandomForestRegressor(n_estimators=100, random_state=42)
    reg_D = RandomForestRegressor(n_estimators=100, random_state=42)
    clf_emotion = RandomForestClassifier(n_estimators=100, random_state=42)

    reg_V.fit(X_fold_train, V_fold_train)
    reg_A.fit(X_fold_train, A_fold_train)
    reg_D.fit(X_fold_train, D_fold_train)
    clf_emotion.fit(X_fold_train, y_fold_train)

    # Predictions for validation fold
    V_pred = reg_V.predict(X_fold_val).reshape(-1, 1)
    A_pred = reg_A.predict(X_fold_val).reshape(-1, 1)
    D_pred = reg_D.predict(X_fold_val).reshape(-1, 1)
    probs = clf_emotion.predict_proba(X_fold_val)

    meta_train[val_idx] = np.hstack([V_pred, A_pred, D_pred, probs])

# --- Train final base models for test prediction ---
reg_V_final = RandomForestRegressor(n_estimators=100, random_state=42)
reg_A_final = RandomForestRegressor(n_estimators=100, random_state=42)
reg_D_final = RandomForestRegressor(n_estimators=100, random_state=42)
clf_emotion_final = RandomForestClassifier(n_estimators=100, random_state=42)

reg_V_final.fit(X_train, V_train)
reg_A_final.fit(X_train, A_train)
reg_D_final.fit(X_train, D_train)
clf_emotion_final.fit(X_train, y_train)

# Predict meta-features for test set
V_pred_test = reg_V_final.predict(X_test).reshape(-1, 1)
A_pred_test = reg_A_final.predict(X_test).reshape(-1, 1)
D_pred_test = reg_D_final.predict(X_test).reshape(-1, 1)
probs_test = clf_emotion_final.predict_proba(X_test)

meta_test = np.hstack([V_pred_test, A_pred_test, D_pred_test, probs_test])

# --- Train and evaluate meta-classifier ---
meta_clf = RandomForestClassifier(n_estimators=100, random_state=42)
meta_clf.fit(meta_train, y_train)
y_pred_meta = meta_clf.predict(meta_test)

# --- Results ---
print("Meta-Classifier Performance:\n")
print(classification_report(y_test, y_pred_meta, target_names=le.classes_))

Meta-Classifier Performance:

              precision    recall  f1-score   support

   Amusement       0.99      0.99      0.99        75
       Anger       0.91      0.89      0.90        75
     Disgust       0.90      0.96      0.93        75
        Fear       0.90      0.92      0.91        75
 Inspiration       0.88      0.85      0.86        75
         Joy       0.97      0.92      0.95        75
     Neutral       0.94      0.92      0.93       100
     Sadness       0.82      0.85      0.84        75
  Tenderness       0.96      0.95      0.95        75

    accuracy                           0.92       700
   macro avg       0.92      0.92      0.92       700
weighted avg       0.92      0.92      0.92       700

